# 4. Research Agent

The **ResearchAgent** takes a single sub-task and produces structured findings. It has access to tools (web search, file I/O, HTTP APIs) to gather information.

Output is a `ResearchFinding` with a summary, facts list, and sources.

In [ ]:
from unittest.mock import AsyncMock, MagicMock
import json

from mares.agents.research_agent import ResearchAgent
from mares.models.sub_task import SubTask

mock_factory = MagicMock()
mock_factory.generate = AsyncMock()
mock_factory.generate.return_value = json.dumps({
    "summary": "Celery uses Redis/RabbitMQ as broker. Failures often stem from connection timeouts or worker crashes.",
    "facts": [
        "Celery supports Redis, RabbitMQ, and SQS brokers",
        "Worker prefetch can cause uneven load distribution",
        "Late acknowledgement can mask failures"
    ],
    "sources": ["https://docs.celeryq.dev"]
})

agent = ResearchAgent(llm_factory=mock_factory)
subtask = SubTask(id=1, task="Research Celery architecture")

finding = await agent.run(subtask)
print(f"Summary: {finding.summary[:80]}...")
print(f"Facts ({len(finding.facts)}):")
for f in finding.facts:
    print(f"  • {f}")
print(f"Sources: {finding.sources}")

## Tool Integration

The research agent has a `ToolExecutor` that can run web searches, read files, and make HTTP requests. Tools are registered centrally and dispatched by name.

In [ ]:
# Show available tools
from mares.tools.tool_registry import default_registry

for tool in default_registry.describe():
    print(f"  {tool['name']}: {tool.get('description', '')[:60]}")

## Research Finding Schema

Each research result includes:
- **summary** — one-paragraph synthesis
- **facts** — bullet-point list of specific findings
- **sources** — URLs or references

In [ ]:
from mares.models.research_finding import ResearchFinding

print("ResearchFinding fields:")
for name, field in ResearchFinding.model_fields.items():
    print(f"  {name}: {field.annotation}")